# IndoBERT Clickbait Detection: `large` model

Trains and evaluates the **large** variant end to end on **Colab, on a T4 or an A100 GPU**: 5 domain specialists
(`Education`, `Health`, `Politics`, `Sport`, `Technology`), then in-domain, cross-domain and
semantic-perturbation evaluation. Run the cells top to bottom.

**Experiment order.** Run `run_base.ipynb`, `run_large.ipynb` and `run_lite.ipynb` (any order, but one at a time
is safest), all with the **same `DRIVE_BASE`**. Then run `run_bayesian_statistics.ipynb`.
The Bayesian analysis compares the models with each other, so it is **not** part of this notebook.

| Step | What happens | Flag |
|---|---|---|
| 0 | Config, Drive, clone `gredss/responsible-ai-toolkit`, GPU check (T4 or A100) | |
| 1 | Unit tests, then GPU smoke test with `large` weights (about 2 min) | `RUN_UNIT_TESTS`, `RUN_SMOKE_TEST` |
| 2 | Train 5 specialists | `RUN_TRAINING` |
| 3 | Evaluate (in-domain, cross-domain, perturbation) | `RUN_EVALUATION` |
| 4 | Result tables and plots | |

**GPU:** works on a T4 (16 GB; fp16 + GradScaler) and on an A100 (bf16); the precision is chosen automatically. The T4 is slower, and `large` may need `TRAIN_BATCH_SIZE = 16` there. Use the same `TRAIN_BATCH_SIZE` (leave it `None`) for all models so results stay comparable.

Safe to re-run: finished training/evaluation is skipped. If Colab disconnects, reconnect and run from the top.

## 0. Configuration

In [ ]:
# ── EDIT ONLY THIS CELL ──────────────────────────────────────────────────────
MODEL      = "large"
REPO_URL   = "https://github.com/gredss/responsible-ai-toolkit.git"
BRANCH     = "main"

# Everything (checkpoints, outputs, results, statistics, logs) is saved directly inside DRIVE_BASE.
# All four notebooks must use the same DRIVE_BASE: they then share data_splits/,
# and the statistics notebook finds every model's results.
# To start a fresh experiment, point DRIVE_BASE to a new folder (existing files are reused, not overwritten).
USE_DRIVE  = True
DRIVE_BASE = "/content/drive/MyDrive/Colab Notebooks/thesis-grace"
SEED       = 42

RUN_UNIT_TESTS = True
RUN_SMOKE_TEST = True
RUN_TRAINING   = True
RUN_EVALUATION = True

GRID_SEARCH   = False      # True = 12 configs x 5 domains (very slow)
AMP_MODE      = "auto"     # auto = bf16 on A100, fp16 + GradScaler on T4 | bf16 | fp16 | off
TRAIN_BATCH_SIZE = None    # None = 32 (default, identical on T4 and A100). Set 16 only if the T4 runs out of memory
DEBUG         = False      # verbose pipeline debug logging
FORCE_RETRAIN = False
FORCE_REEVAL  = False
# ─────────────────────────────────────────────────────────────────────────────

## 1. Environment

In [ ]:
import os, sys, json, time, shutil, subprocess, glob
os.environ["HF_HUB_DISABLE_XET"] = "1"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = DRIVE_BASE
else:
    ROOT = "/content/thesis_out"

REPO    = "/content/responsible-ai-toolkit"
CKPT    = f"{ROOT}/checkpoints"          # <CKPT>/<model>/<Domain>/best_model.pt
OUT     = f"{ROOT}/outputs"              # data_splits/ (shared by all models and by evaluation)
RESULTS = f"{ROOT}/results"              # <RESULTS>/<model>/complete_evaluation.json + perturbed CSVs
STATS   = f"{ROOT}/statistical_analysis"
LOGS    = f"{ROOT}/logs"
LOCAL_LOGS = "/content/_logs"            # logs are written locally, then copied to Drive (Drive is slow for streaming)
DOMAINS = ["Education", "Health", "Politics", "Sport", "Technology"]
ALL_MODELS = ["base", "large", "lite"]

for d in (CKPT, OUT, RESULTS, STATS, LOGS, LOCAL_LOGS):
    os.makedirs(d, exist_ok=True)

def eval_json(model): return f"{RESULTS}/{model}/complete_evaluation.json"
def ckpt_complete(model): return all(os.path.exists(f"{CKPT}/{model}/{d}/best_model.pt") for d in DOMAINS)

print("ROOT:", ROOT)

for name, p in (("checkpoints", CKPT), ("outputs", OUT), ("results", RESULTS)):
    print(f"  {name:<12} {p}  ({sum(len(f) for _, _, f in os.walk(p))} existing files)")

In [ ]:
# Clone (or refresh) the code from GitHub. Always uses the latest commit of BRANCH.
if os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)

print("Commit:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"],
                                capture_output=True, text=True).stdout.strip())

# Guard: fail early if GitHub still holds the old code.
need = {"src/train_pipeline.py": "--amp-mode", "src/evaluate_pipeline.py": "--skip-statistics",
        "src/smoke_test.py": "SMOKE TEST"}
for rel, marker in need.items():
    p = os.path.join(REPO, rel)
    if not os.path.exists(p) or marker not in open(p, encoding="utf-8").read():
        raise RuntimeError(f"{rel} in GitHub '{BRANCH}' is outdated (missing '{marker}'). "
                           "Commit and push your latest local changes, then re-run this cell.")
for rel in ("dataset/data/dict.json", "dataset/data/technology.csv"):
    assert os.path.exists(os.path.join(REPO, rel)), f"missing {rel} in the cloned repo"
print("Code check OK")

In [ ]:
%pip install -q PySastrawi pytest

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU or A100 GPU, then re-run.")
props = torch.cuda.get_device_properties(0)
cc = torch.cuda.get_device_capability(0)
mem_gb = props.total_memory / 1024**3
native_bf16 = cc[0] >= 8                       # A100 yes, T4 no
amp_used = {"auto": "bf16" if native_bf16 else "fp16 + GradScaler", "bf16": "bf16", "fp16": "fp16 + GradScaler", "off": "fp32"}[AMP_MODE]
print(f"GPU: {props.name} | compute capability {cc[0]}.{cc[1]} | {mem_gb:.0f} GB")
print(f"Mixed precision that will be used: {amp_used}  (AMP_MODE = '{AMP_MODE}')")
print(f"Disk free (local /content): {shutil.disk_usage('/content').free / 1024**3:.0f} GB | CPU cores: {os.cpu_count()}")
if USE_DRIVE:
    print(f"Drive free: {shutil.disk_usage('/content/drive').free / 1024**3:.0f} GB "
          "(5 checkpoints of this model, each about 1.4 GB for base/lite and more for large)")
if mem_gb < 20:
    print("\nNOTE: 16 GB GPU (T4). base and lite fit at the default batch size. "
          + ("For 'large', if you see CUDA out-of-memory, set TRAIN_BATCH_SIZE = 16 in the config cell and re-run from the top." if MODEL == "large" else ""))


In [ ]:
ENV = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONDONTWRITEBYTECODE": "1"}

def run(cmd, name, cwd=REPO):
    """Run a command, stream its output here, save a log to Drive, and raise if it fails."""
    stamp = time.strftime("%Y%m%d_%H%M%S")
    local_log = f"{LOCAL_LOGS}/{name}_{stamp}.log"
    print(f"$ {' '.join(cmd)}\n  log -> {LOGS}/{name}_{stamp}.log\n")
    t0 = time.time()
    try:
        with open(local_log, "w") as log:
            p = subprocess.Popen(cmd, cwd=cwd, env=ENV, stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in p.stdout:
                sys.stdout.write(line); log.write(line)
            rc = p.wait()
    finally:
        shutil.copy(local_log, f"{LOGS}/")
    print(f"\n[{name}] exit code {rc} after {(time.time() - t0) / 60:.1f} min")
    if rc != 0:
        raise RuntimeError(f"{name} failed (exit {rc}). See {LOGS}/{name}_{stamp}.log")
    return rc

## 2. Unit tests and GPU smoke test
Both must pass before the long runs. They do not touch your Drive data.

In [ ]:
if RUN_UNIT_TESTS:
    run([sys.executable, "-m", "pytest", "src/test_suite.py", "-q", "-p", "no:cacheprovider"], "unit_tests")
else:
    print("Skipped (RUN_UNIT_TESTS = False)")

In [ ]:
if RUN_SMOKE_TEST:
    # Tiny end-to-end run of THIS model on the GPU: train, checkpoint, evaluate, perturb.
    run([sys.executable, "src/smoke_test.py", "--device", "cuda", "--rows", "24", "--model", MODEL],
        f"smoke_test_{MODEL}")
else:
    print("Skipped (RUN_SMOKE_TEST = False)")

## 3. Training
The train/val/test split is created once in `outputs/data_splits` and reused by every model and by evaluation.

In [ ]:
if RUN_TRAINING:
    if ckpt_complete(MODEL) and not FORCE_RETRAIN:
        print(f"[{MODEL}] all 5 checkpoints already exist, skipping (FORCE_RETRAIN = True to retrain)")
    else:
        cmd = [sys.executable, "src/train_pipeline.py", "--model", MODEL,
               "--dataset-dir", "dataset/data", "--output-dir", OUT, "--checkpoint-dir", CKPT,
               "--device", "cuda", "--amp-mode", AMP_MODE, "--seed", str(SEED)]
        if TRAIN_BATCH_SIZE: cmd += ["--batch-size", str(TRAIN_BATCH_SIZE)]
        if GRID_SEARCH: cmd.append("--grid-search")
        if DEBUG:       cmd.append("--debug")
        run(cmd, f"train_{MODEL}")
        if not ckpt_complete(MODEL):
            raise RuntimeError(f"training finished but some best_model.pt files are missing in {CKPT}/{MODEL}")
        print(f"[{MODEL}] OK: 5 specialists saved")
else:
    print("Skipped (RUN_TRAINING = False)")

In [ ]:
# Checkpoints and best validation score per specialist
import pandas as pd
import matplotlib.pyplot as plt
rows = []
for d in DOMAINS:
    p, h = f"{CKPT}/{MODEL}/{d}/best_model.pt", f"{CKPT}/{MODEL}/{d}/training_history.json"
    row = {"domain": d, "checkpoint": os.path.exists(p),
           "size_MB": round(os.path.getsize(p) / 1024**2) if os.path.exists(p) else None}
    if os.path.exists(h):
        f1 = json.load(open(h))["val_macro_f1"]
        row.update(best_val_macro_f1=round(max(f1), 4), best_epoch=f1.index(max(f1)) + 1, epochs_run=len(f1))
    rows.append(row)
display(pd.DataFrame(rows).set_index("domain"))

fig, axes = plt.subplots(1, len(DOMAINS), figsize=(4 * len(DOMAINS), 3), sharey=True)
for ax, d in zip(axes, DOMAINS):
    h = f"{CKPT}/{MODEL}/{d}/training_history.json"
    if not os.path.exists(h):
        ax.set_title(f"{d} (missing)"); continue
    hist = json.load(open(h))
    ax.plot(hist["epochs"], hist["train_loss"], marker="o", label="train")
    ax.plot(hist["epochs"], hist["val_loss"], marker="o", label="val")
    ax.set_title(d); ax.set_xlabel("epoch"); ax.grid(alpha=.3)
axes[0].set_ylabel("loss"); axes[0].legend()
fig.suptitle(f"Training curves: {MODEL}", y=1.03); plt.tight_layout(); plt.show()

## 4. Evaluation
In-domain, 5x5 cross-domain, semantic perturbation (low/medium/high) and cross-domain x perturbation. The Bayesian step is skipped here (`--skip-statistics`).

In [ ]:
if RUN_EVALUATION:
    if os.path.exists(eval_json(MODEL)) and not FORCE_REEVAL:
        print(f"[{MODEL}] complete_evaluation.json already exists, skipping (FORCE_REEVAL = True to redo)")
    else:
        if not ckpt_complete(MODEL):
            raise RuntimeError(f"[{MODEL}] checkpoints incomplete in {CKPT}/{MODEL}; train first")
        cmd = [sys.executable, "src/evaluate_pipeline.py", "--model", MODEL,
               "--checkpoint-dir", CKPT, "--dataset-dir", "dataset/data",
               "--output-dir", RESULTS, "--train-output-dir", OUT, "--perturbation-output-dir", RESULTS,
               "--device", "cuda", "--amp-mode", AMP_MODE, "--seed", str(SEED), "--skip-statistics"]
        if DEBUG: cmd.append("--debug")
        run(cmd, f"eval_{MODEL}")
        if not os.path.exists(eval_json(MODEL)):
            raise RuntimeError(f"evaluation finished but {eval_json(MODEL)} was not written")
        print(f"[{MODEL}] OK")
else:
    print("Skipped (RUN_EVALUATION = False)")

## 5. Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
R = json.load(open(eval_json(MODEL), encoding="utf-8"))

# In-domain
rows = []
for d, v in R["in_domain"].items():
    x = v["metrics"]
    rows.append({"domain": d, "n": v["num_samples"], "accuracy": x["accuracy"], "macro_f1": x["macro_f1"],
                 "f1_non_clickbait": x["f1_0"], "f1_clickbait": x["f1_1"], "mcc": x["mcc"]})
print("IN-DOMAIN"); display(pd.DataFrame(rows).round(4).set_index("domain"))

# Cross-domain Macro-F1 (row = source specialist, column = target test set)
doms = list(R["in_domain"].keys())
mat = pd.DataFrame({t: {s: R["cross_domain"][f"{s}->{t}"]["metrics"]["macro_f1"] for s in doms} for t in doms})
print("\nCROSS-DOMAIN Macro-F1 (row = source, column = target)"); display(mat.round(4))

In [ ]:
# Semantic perturbation: Macro-F1 per level on the matched subset, and prediction-flip rate
LEVELS = ["low", "medium", "high"]
rows = []
for d, blk in R["perturbation"].items():
    clean = blk["clean"]["metrics"]["macro_f1"]
    row = {"domain": d, "n_matched": blk["clean"].get("common_ids_count"), "clean": clean}
    for lv in LEVELS:
        b = blk["semantic"][lv]
        row[lv] = b["metrics"]["macro_f1"]
        row[f"flip_{lv}_%"] = b["prediction_flip"]["flip_rate_pct"]
    row["drop_high"] = clean - row["high"]
    rows.append(row)
pert_df = pd.DataFrame(rows).set_index("domain")
display(pert_df.round(4))

fig, ax = plt.subplots(figsize=(6, 4))
for d, row in pert_df.iterrows():
    ax.plot(["clean"] + LEVELS, [row["clean"]] + [row[lv] for lv in LEVELS], marker="o", label=d)
ax.set_title(f"{MODEL}: semantic perturbation"); ax.set_xlabel("intensity"); ax.set_ylabel("Macro-F1")
ax.grid(alpha=.3); ax.legend(fontsize=8); plt.tight_layout()
fig.savefig(f"{RESULTS}/{MODEL}/perturbation_curves.png", dpi=150); plt.show()

in_f1 = [v["metrics"]["macro_f1"] for v in R["in_domain"].values()]
cross = [v["metrics"]["macro_f1"] for k, v in R["cross_domain"].items() if k.split("->")[0] != k.split("->")[1]]
summary = {"model": MODEL, "avg_in_domain_f1": sum(in_f1) / len(in_f1), "avg_cross_domain_f1": sum(cross) / len(cross),
           **{f"avg_f1_{lv}": pert_df[lv].mean() for lv in LEVELS}}
pd.DataFrame([summary]).round(6).to_csv(f"{RESULTS}/{MODEL}/model_summary.csv", index=False)
print("\nSUMMARY", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in summary.items()})

In [ ]:
# Progress across the whole experiment
print(f"Experiment: {ROOT}")
for m in ALL_MODELS:
    print(f"  {m:<6} checkpoints: {'yes' if ckpt_complete(m) else 'no ':<3} | evaluation: {'yes' if os.path.exists(eval_json(m)) else 'no'}")
done = [m for m in ALL_MODELS if os.path.exists(eval_json(m))]
print("\nNext:", "run_bayesian_statistics.ipynb (all three models are evaluated)" if len(done) == 3
      else f"finish the remaining model notebooks ({', '.join(m for m in ALL_MODELS if m not in done)}), then run_bayesian_statistics.ipynb")